# Task
Clean a messy subscriber table: fix types, text, dates and missing values so it can be trusted for analysis.

# Data

In [2]:
import pandas as pd
import numpy as np

In [3]:
raw = pd.DataFrame({
    "subscriber_id": [101, 102, 103, 104, 104, 105, 106, 107],
    "tariff": ["Base", "base ", "PREMIUM", "Premium", "Premium", None, "Base", " Max"],
    "monthly_fee": ["2500", "2 500", "7000", "7000", "7000", "3500", "n/a", "9900"],
    "join_date": ["2025-01-15", "15.02.2025", "2025-03-01", "2025-03-10", "2025-03-10", None, "2025-05-20", "2025-06-01"],
    "city": ["Astana", "astana", "Almaty", "Almaty ", "Almaty ", "Shymkent", None, "ASTANA"],
})
raw.to_csv("subscribers_raw.csv", index=False)

In [4]:
df = pd.read_csv('subscribers_raw.csv')
df.head()

,subscriber_id,tariff,monthly_fee,join_date,city
0,101,Base,2500,2025-01-15,Astana
1,102,base,2 500,15.02.2025,astana
2,103,PREMIUM,7000,2025-03-01,Almaty
3,104,Premium,7000,2025-03-10,Almaty
4,104,Premium,7000,2025-03-10,Almaty


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   subscriber_id  8 non-null      int64
 1   tariff         7 non-null      str  
 2   monthly_fee    7 non-null      str  
 3   join_date      7 non-null      str  
 4   city           7 non-null      str  
dtypes: int64(1), str(4)
memory usage: 452.0 bytes


In [6]:
df = df.drop_duplicates(subset='subscriber_id', keep='first')
print(df.shape)

df['monthly_fee'] = df['monthly_fee'].str.replace(' ','')
df['monthly_fee'] = pd.to_numeric(df['monthly_fee'], errors='coerce')
df['monthly_fee']

(7, 5)


0    2500.0
1    2500.0
2    7000.0
3    7000.0
5    3500.0
6       NaN
7    9900.0
Name: monthly_fee, dtype: float64

In [7]:
print(df.shape)
df['subscriber_id'].value_counts()

(7, 5)


subscriber_id
101    1
102    1
103    1
104    1
105    1
106    1
107    1
Name: count, dtype: int64

In [8]:
print('Tariff before:', df['tariff'].unique())
df['tariff'] = df['tariff'].str.strip().str.title()
print('Tariff after:', df['tariff'].unique())

print('')

print('City before:', df['city'].unique())
df['city'] = df['city'].str.strip().str.title()
print('City after:', df['city'].unique())


Tariff before: <StringArray>
['Base', 'base ', 'PREMIUM', 'Premium', nan, ' Max']
Length: 6, dtype: str
Tariff after: <StringArray>
['Base', 'Premium', nan, 'Max']
Length: 4, dtype: str

City before: <StringArray>
['Astana', 'astana', 'Almaty', 'Almaty ', 'Shymkent', nan, 'ASTANA']
Length: 7, dtype: str
City after: <StringArray>
['Astana', 'Almaty', 'Shymkent', nan]
Length: 4, dtype: str


In [9]:
iso = pd.to_datetime(df['join_date'], format="%Y-%m-%d", errors='coerce')
dot = pd.to_datetime(df['join_date'], format="%d.%m.%Y", errors='coerce')

df['join_date'] = iso.fillna(dot)
df[['subscriber_id', 'join_date']]

,subscriber_id,join_date
0,101,2025-01-15
1,102,2025-02-15
2,103,2025-03-01
3,104,2025-03-10
5,105,NaT
6,106,2025-05-20
7,107,2025-06-01


In [10]:
df.info()

<class 'pandas.DataFrame'>
Index: 7 entries, 0 to 7
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   subscriber_id  7 non-null      int64         
 1   tariff         6 non-null      str           
 2   monthly_fee    6 non-null      float64       
 3   join_date      6 non-null      datetime64[us]
 4   city           6 non-null      str           
dtypes: datetime64[us](1), float64(1), int64(1), str(2)
memory usage: 336.0 bytes


In [11]:
print(df.isna().sum())

df['tariff'] = df['tariff'].fillna('Unknown')
df['city'] = df['city'].fillna('Unknown')

df['fee_missing'] = df['monthly_fee'].isna()
df['monthly_fee'] = df['monthly_fee'].fillna(df['monthly_fee'].median())

df = df.reset_index(drop=True)
df

subscriber_id    0
tariff           1
monthly_fee      1
join_date        1
city             1
dtype: int64


,subscriber_id,tariff,monthly_fee,join_date,city,fee_missing
0,101,Base,2500.0,2025-01-15,Astana,False
1,102,Base,2500.0,2025-02-15,Astana,False
2,103,Premium,7000.0,2025-03-01,Almaty,False
3,104,Premium,7000.0,2025-03-10,Almaty,False
4,105,Unknown,3500.0,NaT,Shymkent,False
5,106,Base,5250.0,2025-05-20,Unknown,True
6,107,Max,9900.0,2025-06-01,Astana,False


In [12]:
df.loc[df["fee_missing"], "monthly_fee"] = np.nan

df["monthly_fee"] = df["monthly_fee"].fillna(
    df.groupby("tariff")["monthly_fee"].transform("median")
)

df.loc[df["fee_missing"], ["subscriber_id", "tariff", "monthly_fee"]]

,subscriber_id,tariff,monthly_fee
5,106,Base,2500.0


- Raw: 8 rows, all columns text except id; 7 problems found
- Fixed: 1 duplicate (by key), fee "2 500" → number, 2 date formats → datetime,
  tariff 5 spellings → 3, city 6 spellings → 3, missing text → "Unknown"
- Fee for missing value: overall median 5250 vs tariff median 2500 → tariff median is correct
- join_date left empty: it's a fact, not an estimate

## 5. What I learned
1. Typical export problems and fixes: duplicates → drop_duplicates(subset=key); "2 500" → str.replace + to_numeric;
   mixed date formats → to_datetime with an explicit format for each; messy text → str.strip().str.title().
2. errors="coerce" silently turns bad values into NaN — always check isna().sum() before and after.
3. Fill estimates, not facts: fee filled by the median of its own tariff (2500), not the overall median (5250);
   join_date stays empty. A flag column keeps track of what was filled.